# D1 local ablation (qwen3.5:4b via Ollama) on Colab

Runs `scripts/local_ablation.py` — the V1 local tier arm `D1_local_single` — against Ollama's `qwen3.5:4b` inside a Colab runtime instead of your laptop.

**Before running:** Runtime → Change runtime type → pick a GPU (T4 is fine, free tier). CPU-only will also work, just slower.

Only the `D1` arm is implemented locally today (single local agent). The hosted `A_deterministic` / `C_crew_llm` arms are a separate, not-yet-built piece of work — out of scope here.

Order: install Ollama → pull the model → get the repo onto the runtime → install Python deps → fetch external corpora → build the dev manifest → `freeze` → `run` → `summarise`.

In [ ]:
!apt-get update && apt-get install -y zstd

## 1. Install and start Ollama

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time, urllib.request

def daemon_up() -> bool:
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/version', timeout=2)
        return True
    except Exception:
        return False

if daemon_up():
    print('ollama daemon already up')
else:
    # Detached, logging to a file: a daemon whose stdout/stderr go to an unread PIPE
    # blocks once the buffer fills, and one tied to this cell dies with the kernel.
    log = open('/content/ollama.log', 'ab')
    subprocess.Popen(['ollama', 'serve'], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    for attempt in range(120):
        if daemon_up():
            print(f'ollama daemon is up (after {attempt + 1} s)')
            break
        time.sleep(1)
    else:
        raise SystemExit('ollama daemon did not start; see /content/ollama.log')

# Re-run this cell after any runtime reset or reconnect: freeze/run refuse without the daemon.
!ollama ps


In [ ]:
!ollama pull qwen3.5:4b

## 2. Get the repo onto the runtime

The repo is public (`shayb1187-a11y/agentic-threat-hunter`), so a plain `git clone` is enough. Two options — use whichever is easier:

**Option A — clone.** Run the cell below. No token, no Git LFS: the repository tracks nothing with LFS.

**Option B — upload a zip.** Zip your local working copy (skip `.git`, `data/external/`, `reports/local/`) and upload/mount it, then skip the clone cell and just `%cd` into the extracted folder.


In [ ]:
# Option A: plain clone of the public repository
REPO = "shayb1187-a11y/agentic-threat-hunter"
BRANCH = "m14-real-data-validation"

!git clone -b {BRANCH} https://github.com/{REPO}.git /content/agentic-threat-hunter


In [ ]:
%cd /content/agentic-threat-hunter

If you're continuing a run across sessions, point this at Drive instead so `reports/local/dev/rows/` (already-completed rows are skipped, not re-run) and `data/external/` (large corpora, slow to refetch) survive a runtime reset:
```python
from google.colab import drive
drive.mount('/content/drive')
# then symlink or copy reports/local and data/external to/from /content/drive/MyDrive/...
```

## 3. Python deps

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .

## 4. Fetch external corpora + build the dev manifest

`data/external/` is gitignored (only the manifest index is committed) so it has to be fetched fresh on a new runtime.

The dev build only needs `flaws_cloud` (for the `flaws_cloud` dev bundle) plus the DEDALE D02 Winlogbeat hour (for the ten injected dev cases) — fetch those directly rather than `fetch_external.py`'s full manifest, since that also includes `k8s_ci`, whose pinned GCS log artifact has since expired (test-infra garbage-collects old CI logs) and would abort the whole fetch with a 404.

In [ ]:
!python scripts/fetch_external.py flaws_cloud

In [ ]:
!python scripts/dedale_fetch_hours.py --days 2

In [ ]:
!python scripts/local_inject_dedale.py

In [ ]:
!python scripts/local_manifest.py build

## 5. freeze / smoke / run / summarise (D1 v3, the bounded investigator)

`freeze` pins the model digest, daemon version, the investigator's prompt hashes and bounds against the manifest hash; `run` refuses if any of them drifted. Rows are written under `reports/local/dev/rows/D1_qwen3.5-4b/m<manifest12>_d1-investigator-v3/` and the row key carries the manifest hash, so rows from another manifest or another prompt version are never mixed. `run` is resumable: rerunning after a disconnect skips rows already on disk.

**Manifest hash caveat.** The telemetry digest depends on the runtime (numpy/pandas/Python): the laptop builds `2298b0e8`, this Colab runtime builds `e4115893`, for byte-identical cases. Build, freeze, run and summarise must all happen on this runtime, which the cells above and below do.

**Smoke first.** Five representative cases (V1 obvious malicious + lineage discovery, V2 subtle malicious, V7 and V8 benign look-alikes, flaws CASE-071 ambiguous). Read the per-case investigation table before launching the full twenty: tool choices should differ when gaps differ, benign cases may stay benign, hypotheses must not copy the detector, new evidence ids should appear, and no call should hit the 768-token cap. If that fails, stop and change the investigator (a new prompt version = a new commit + re-freeze), do not tune on the full set.

**What this run measures.** v3 (`59fadc1`) is the second of at most three tuning passes and has not been measured yet; running it as-is costs no pass. Read `LINK-2 recovered / defined` from the investigation table: the harness recovers it 9/9 under the offline oracle replay (`python scripts/local_replay.py --path link`, no model), so a zero here belongs to the model. Each round now records the raw model reply and the prompt's sha256, so a downloaded row can be replayed offline and its prompts matched byte for byte.

In [ ]:
!python scripts/local_ablation.py freeze --model qwen3.5:4b --arm D1

In [ ]:
!python scripts/local_ablation.py run --model qwen3.5:4b --arm D1 --repeat 1 --seed 0 --only dedale_injected_dev:V1/CASE-001 dedale_injected_dev:V2/CASE-001 dedale_injected_dev:V7/CASE-001 dedale_injected_dev:V8/CASE-001 flaws_cloud/CASE-071

In [ ]:
!python scripts/local_ablation.py summarise --model qwen3.5:4b --arm D1 --repeat 1 > /dev/null
text = open('reports/local/dev/SUMMARY_D1_qwen3.5-4b_rep1.md').read()
print(text[text.index('## Per case: investigation'):])

In [ ]:
# Every model claim of the smoke rows, to check they are not detector paraphrases
import glob, json
for path in sorted(glob.glob('reports/local/dev/rows/D1_qwen3.5-4b/*/*.json')):
    row = json.load(open(path))['row']; inv = row['state'].get('investigation', {})
    print('=' * 90); print(row['corpus'], row['case_id'], '| label', row.get('labels', {}).get('verdict', 'unlabelled'), '| disposition', inv.get('final_disposition'), '| probes', inv.get('probes_run'), '| truncated', inv.get('output_truncated'))
    print('  gap:', inv.get('evidence_gap')); print('  reason:', inv.get('tool_choice_reason'))
    for c in row['state']['claims']:
        if c['source'] == 'llm': print('  ', c['type'], c['statement'][:200], c['evidence_ids'])
    print('  rejected:', [r['reason'][:70] for r in row['state']['rejected_claims']], '| links:', row.get('label_scores', {}).get('links'))
    for r in inv.get('rounds', []):
        print('  round', r.get('round'), '| chose', (r.get('chosen_probe') or {}).get('tool'), '| new ids', r.get('new_evidence_ids_returned'), '| prompt', str(r.get('prompt_sha256'))[:12], r.get('prompt_chars'), 'chars')
        print('    raw:', (r.get('raw') or '')[:400])

### Full run (only after the smoke table is sane)

The five smoke rows are ordinary rows under the same identity and are skipped, so the full run adds the remaining fifteen.

In [ ]:
!python scripts/local_ablation.py run --model qwen3.5:4b --arm D1 --repeat 1 --seed 0

In [ ]:
!python scripts/local_ablation.py summarise --model qwen3.5:4b --arm D1 --repeat 1

In [ ]:
print(open("reports/local/dev/SUMMARY_D1_qwen3.5-4b_rep1.md").read())

## 6. (optional) pull results back down

Zip `reports/local/dev/` and download it, or copy it to a mounted Drive, so the rows survive the Colab runtime being recycled.

In [ ]:
!zip -r -q /content/dev_results.zip reports/local/dev
from google.colab import files
files.download("/content/dev_results.zip")